<a href="https://colab.research.google.com/github/Joey-Jireh/eye-of-ra/blob/main/notebooks/week3/week3_detection_engines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 — Week 3 Header & Library Imports
# Eye of Ra 👁️ — Week 3: Detection Engines & Risk Scoring

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("✅ Libraries loaded — Week 3 begins")
print("👁️ Eye of Ra — Detection Engines & Risk Scoring")

# Git: Cell 1 — Week 3 header and imports

✅ Libraries loaded — Week 3 begins
👁️ Eye of Ra — Detection Engines & Risk Scoring


In [2]:
# Cell 2 — Load dataset and model
df = pd.read_csv('/content/eye_of_ra_master_dataset_v3.csv')
model = joblib.load('/content/eye_of_ra_model_v1.pkl')
feature_cols = joblib.load('/content/eye_of_ra_features_v1.pkl')

print(f"✅ Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"✅ Model loaded: {type(model).__name__}")
print(f"✅ Features loaded: {len(feature_cols)} features")
print(f"\nFeatures in model:")
for f in feature_cols:
    print(f"  - {f}")
print(f"\nFraud labels: {df['fraud_label'].sum()} | Clean: {(df['fraud_label']==0).sum()}")

# Git: Cell 2 — load dataset and model for Week 3

✅ Dataset loaded: 262 rows x 21 columns
✅ Model loaded: XGBClassifier
✅ Features loaded: 12 features

Features in model:
  - is_variation
  - is_sole_source
  - entity_frequency
  - supplier_frequency
  - repeat_supplier
  - audit_flagged
  - audit_flag_count
  - award_concentration
  - method_abuse_score
  - supplier_risk_score
  - audit_intensity
  - variation_abuse_rate

Fraud labels: 23 | Clean: 239


In [3]:
# Cell 3 — Engine 1: Bid Manipulation Detector
# Grounded in Act 663 S.38-41 (sole source rules) and S.59 (bid rigging)

def bid_manipulation_score(row, df):
    score = 0
    flags = []

    # Signal 1 — Sole source contract (S.38-41: restricted to genuine emergencies)
    if row['is_sole_source'] == 1:
        score += 30
        flags.append("Sole source procurement used (S.38-41 Act 663)")

    # Signal 2 — Contract variation on already suspicious contract
    if row['is_variation'] == 1 and row['audit_flagged'] == 1:
        score += 25
        flags.append("Variation issued on audit-flagged contract (S.59 Act 663)")

    # Signal 3 — Supplier has high award concentration (dominates awards)
    if row['award_concentration'] > 0.3:
        score += 20
        flags.append(f"Supplier holds {row['award_concentration']*100:.1f}% of entity awards — concentration risk")

    # Signal 4 — Method abuse score already elevated
    if row['method_abuse_score'] > 0.5:
        score += 15
        flags.append("Procurement method abuse pattern detected (S.40-41 Act 663)")

    # Signal 5 — Variation abuse rate is high for this entity
    if row['variation_abuse_rate'] > 0.3:
        score += 10
        flags.append(f"Entity variation abuse rate: {row['variation_abuse_rate']*100:.1f}%")

    return min(score, 100), flags  # cap at 100

# Run on full dataset
results = df.apply(lambda row: bid_manipulation_score(row, df), axis=1)
df['engine1_score'] = results.apply(lambda x: x[0])
df['engine1_flags'] = results.apply(lambda x: x[1])

# Summary
flagged = df[df['engine1_score'] > 0]
high_risk = df[df['engine1_score'] >= 50]

print("=" * 55)
print("👁️  ENGINE 1 — BID MANIPULATION DETECTOR")
print("=" * 55)
print(f"Contracts scanned:     {len(df)}")
print(f"Contracts flagged:     {len(flagged)}")
print(f"High risk (score≥50):  {len(high_risk)}")
print(f"\nTop 10 highest risk contracts:")
print(df[['entity', 'supplier', 'engine1_score']].sort_values('engine1_score', ascending=False).head(10).to_string(index=True))

# Git: Cell 3 — Engine 1 bid manipulation detector

👁️  ENGINE 1 — BID MANIPULATION DETECTOR
Contracts scanned:     262
Contracts flagged:     205
High risk (score≥50):  42

Top 10 highest risk contracts:
                                                                            entity                        supplier  engine1_score
247                                                        Ministry of Information                         Unknown             75
198                          Controller and Accountant General’s Department (CAGD)      Netsolutions Ghana Limited             75
119                                                      Ministry of Finance (MoF)                         Unknown             75
207                                                  Youth Employment Agency (YEA)                         Unknown             75
200                                            Ministry of Youth and Sports (MOYS)                         Unknown             75
249  Ministry of Local Government, Decentralization and Rural Devel